# 열처리 공정 Digital Twin — 일반인 이해용 영상 생성

이 Notebook은 **NVIDIA Isaac Sim 없이 Google Colab에서 실행**됩니다.

목적은 열처리 공정을 모르는 사람도 다음 개념을 영상으로 쉽게 이해하도록 하는 것입니다.

- 실제 열처리로와 가상 Digital Twin의 연결
- 제품 장입 → 승온 → 유지 → 분위기 제어 → 확산 → 냉각 → 완료
- 온도, 히터 출력, 탄소 포텐셜, 에너지, 탄소배출 변화
- Digital Twin의 역할: 상태 재현, 미래 예측, What-if 비교, 최적 조건 추천
- 최종 결과를 MP4로 저장

### 출력 파일
- `heat_treatment_digital_twin_explainer.mp4`
- 핵심 장면 PNG 10장


In [ ]:
# Colab 환경 준비
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install matplotlib numpy


In [ ]:
# Python 프로그램 생성
from pathlib import Path

PROGRAM_PATH = Path("/content/heat_treatment_digital_twin_layperson.py")

PROGRAM = r'''
# -*- coding: utf-8 -*-
"""
Heat Treatment Digital Twin - Layperson-Friendly Animation
Google Colab compatible.

Purpose
-------
This program does NOT require NVIDIA Isaac Sim.
It creates an easy-to-understand digital twin explainer video for a heat-treatment process.

The video shows:
1) physical furnace and workpieces
2) temperature rise / hold / cooling
3) heater power, carbon potential, energy usage
4) quality risk and anomaly risk
5) Digital Twin prediction and operator decision concept
6) final energy / carbon / quality outcome

Output:
- heat_treatment_digital_twin_explainer.mp4
- 10 key-scene PNG images
"""

from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

ROOT = Path("/content/heat_treatment_digital_twin_demo")
FRAME_DIR = ROOT / "key_scenes"
ROOT.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)

FPS = 24
DURATION_SEC = 30
N_FRAMES = FPS * DURATION_SEC

# ------------------------------------------------------------
# 1) 10 key process stages
# ------------------------------------------------------------
STAGES = [
    {
        "name": "1. 공정 시작",
        "short": "START",
        "progress": 0.00,
        "temp": 25,
        "target": 850,
        "heater": 0,
        "cp": 0.00,
        "energy": 0,
        "carbon": 0,
        "quality_risk": 0.02,
        "anomaly_risk": 0.01,
        "tray_x": -2.8,
        "message": "실제 열처리 공정의 상태를 Digital Twin이 함께 따라갑니다."
    },
    {
        "name": "2. 제품 장입",
        "short": "LOADING",
        "progress": 0.10,
        "temp": 80,
        "target": 850,
        "heater": 10,
        "cp": 0.00,
        "energy": 3,
        "carbon": 1,
        "quality_risk": 0.02,
        "anomaly_risk": 0.01,
        "tray_x": -2.0,
        "message": "제품, 장입량, 위치, 레시피 정보를 Heat ID로 묶어 관리합니다."
    },
    {
        "name": "3. 승온 시작",
        "short": "HEATING",
        "progress": 0.22,
        "temp": 350,
        "target": 850,
        "heater": 75,
        "cp": 0.10,
        "energy": 30,
        "carbon": 13,
        "quality_risk": 0.03,
        "anomaly_risk": 0.02,
        "tray_x": -1.3,
        "message": "히터 출력을 높여 제품 온도를 목표 온도까지 올립니다."
    },
    {
        "name": "4. 고온 승온",
        "short": "HEATING",
        "progress": 0.35,
        "temp": 650,
        "target": 850,
        "heater": 92,
        "cp": 0.30,
        "energy": 67,
        "carbon": 29,
        "quality_risk": 0.04,
        "anomaly_risk": 0.03,
        "tray_x": -0.7,
        "message": "Digital Twin은 온도 상승, 전력 사용, 예상 완료 시점을 함께 계산합니다."
    },
    {
        "name": "5. 유지",
        "short": "HOLDING",
        "progress": 0.48,
        "temp": 850,
        "target": 850,
        "heater": 48,
        "cp": 0.55,
        "energy": 104,
        "carbon": 45,
        "quality_risk": 0.03,
        "anomaly_risk": 0.02,
        "tray_x": 0.0,
        "message": "목표 온도를 유지하면서 제품 내부까지 균일한 열처리를 진행합니다."
    },
    {
        "name": "6. 분위기 제어",
        "short": "CARBURIZING",
        "progress": 0.60,
        "temp": 850,
        "target": 850,
        "heater": 52,
        "cp": 0.80,
        "energy": 141,
        "carbon": 62,
        "quality_risk": 0.04,
        "anomaly_risk": 0.03,
        "tray_x": 0.2,
        "message": "탄소 포텐셜과 가스 조건을 관리하여 필요한 표면 특성을 만듭니다."
    },
    {
        "name": "7. 확산",
        "short": "DIFFUSION",
        "progress": 0.70,
        "temp": 830,
        "target": 830,
        "heater": 34,
        "cp": 0.62,
        "energy": 163,
        "carbon": 71,
        "quality_risk": 0.03,
        "anomaly_risk": 0.02,
        "tray_x": 0.7,
        "message": "온도와 시간을 조절해 표면과 내부의 품질 균형을 맞춥니다."
    },
    {
        "name": "8. 사전 냉각",
        "short": "PRE-COOLING",
        "progress": 0.79,
        "temp": 600,
        "target": 600,
        "heater": 8,
        "cp": 0.25,
        "energy": 171,
        "carbon": 74,
        "quality_risk": 0.04,
        "anomaly_risk": 0.03,
        "tray_x": 1.3,
        "message": "냉각 조건에 따라 최종 경도와 변형 위험이 달라질 수 있습니다."
    },
    {
        "name": "9. 소입 / 급냉",
        "short": "QUENCHING",
        "progress": 0.90,
        "temp": 180,
        "target": 100,
        "heater": 0,
        "cp": 0.00,
        "energy": 181,
        "carbon": 79,
        "quality_risk": 0.08,
        "anomaly_risk": 0.05,
        "tray_x": 2.0,
        "message": "급격한 냉각 구간은 품질과 변형에 큰 영향을 주므로 집중 관리합니다."
    },
    {
        "name": "10. 공정 완료",
        "short": "COMPLETE",
        "progress": 1.00,
        "temp": 70,
        "target": 70,
        "heater": 0,
        "cp": 0.00,
        "energy": 184,
        "carbon": 79,
        "quality_risk": 0.02,
        "anomaly_risk": 0.01,
        "tray_x": 2.8,
        "message": "실제 결과와 예측 결과를 비교해 다음 Heat의 운전 조건을 개선합니다."
    },
]

# ------------------------------------------------------------
# 2) Smooth interpolation between stages
# ------------------------------------------------------------
def lerp(a, b, t):
    return a + (b - a) * t

def stage_state(frame_idx):
    x = frame_idx / max(1, N_FRAMES - 1) * (len(STAGES) - 1)
    i0 = int(np.floor(x))
    i1 = min(i0 + 1, len(STAGES) - 1)
    t = x - i0

    a, b = STAGES[i0], STAGES[i1]
    numeric_keys = [
        "progress", "temp", "target", "heater", "cp", "energy",
        "carbon", "quality_risk", "anomaly_risk", "tray_x"
    ]

    s = {}
    for k in numeric_keys:
        s[k] = lerp(a[k], b[k], t)

    # Keep labels stable within a section for readability.
    s["name"] = a["name"] if t < 0.5 else b["name"]
    s["short"] = a["short"] if t < 0.5 else b["short"]
    s["message"] = a["message"] if t < 0.5 else b["message"]
    s["stage_index"] = i0 if t < 0.5 else i1
    return s

# ------------------------------------------------------------
# 3) Drawing helpers
# ------------------------------------------------------------
def draw_bar(ax, x, y, width, value_0_1, label, value_text):
    ax.add_patch(plt.Rectangle((x, y), width, 0.035, fill=False, linewidth=1.2))
    fill_w = max(0.0, min(1.0, value_0_1)) * width
    ax.add_patch(plt.Rectangle((x, y), fill_w, 0.035))
    ax.text(x, y + 0.048, label, fontsize=9, va="bottom")
    ax.text(x + width, y + 0.048, value_text, fontsize=9, va="bottom", ha="right")

def draw_furnace(ax, s):
    # Furnace shell
    ax.add_patch(plt.Rectangle((0.08, 0.34), 0.58, 0.36, fill=False, linewidth=2.0))
    ax.add_patch(plt.Rectangle((0.10, 0.36), 0.54, 0.32, alpha=0.08))

    # Three process zones
    ax.plot([0.28, 0.28], [0.36, 0.68], linestyle="--", linewidth=1.0)
    ax.plot([0.47, 0.47], [0.36, 0.68], linestyle="--", linewidth=1.0)
    ax.text(0.18, 0.665, "가열", fontsize=10, ha="center", va="top")
    ax.text(0.375, 0.665, "유지/확산", fontsize=10, ha="center", va="top")
    ax.text(0.565, 0.665, "냉각", fontsize=10, ha="center", va="top")

    # Simulated thermal field using default matplotlib colormap.
    temp_norm = np.clip((s["temp"] - 25) / 825.0, 0.0, 1.0)
    xs = np.linspace(0.11, 0.63, 160)
    ys = np.linspace(0.37, 0.64, 70)
    xx, yy = np.meshgrid(xs, ys)

    # Center thermal hotspot follows the tray.
    tx = np.interp(s["tray_x"], [-2.8, 2.8], [0.12, 0.62])
    thermal = temp_norm * np.exp(-((xx - tx) ** 2 / 0.015 + (yy - 0.51) ** 2 / 0.05))
    thermal += temp_norm * 0.25

    ax.imshow(
        thermal,
        extent=(0.11, 0.63, 0.37, 0.64),
        origin="lower",
        aspect="auto",
        alpha=0.45,
    )

    # Tray and workpieces
    tray_plot_x = tx
    ax.add_patch(plt.Rectangle((tray_plot_x - 0.065, 0.42), 0.13, 0.03, fill=False, linewidth=1.5))
    part_xs = np.linspace(tray_plot_x - 0.05, tray_plot_x + 0.05, 4)
    for px in part_xs:
        ax.add_patch(plt.Rectangle((px - 0.01, 0.45), 0.02, 0.055, alpha=0.7))

    # Sensor markers
    sensor_positions = [(0.18, 0.61), (0.375, 0.61), (0.565, 0.61)]
    for i, (sx, sy) in enumerate(sensor_positions, start=1):
        ax.scatter([sx], [sy], s=35)
        ax.text(sx, sy + 0.02, f"TC{i}", fontsize=7, ha="center")

    # Simple Digital Twin mirror box
    ax.add_patch(plt.Rectangle((0.72, 0.36), 0.20, 0.34, fill=False, linewidth=1.5))
    ax.text(0.82, 0.675, "DIGITAL TWIN", fontsize=11, ha="center", va="top", fontweight="bold")
    ax.text(0.82, 0.62, "현재 상태 재현", fontsize=9, ha="center")
    ax.text(0.82, 0.575, "미래 상태 예측", fontsize=9, ha="center")
    ax.text(0.82, 0.53, "What-if 비교", fontsize=9, ha="center")
    ax.text(0.82, 0.485, "최적 조건 추천", fontsize=9, ha="center")

    # Connection
    ax.annotate(
        "",
        xy=(0.72, 0.52),
        xytext=(0.66, 0.52),
        arrowprops=dict(arrowstyle="->", linewidth=1.5),
    )
    ax.annotate(
        "",
        xy=(0.66, 0.46),
        xytext=(0.72, 0.46),
        arrowprops=dict(arrowstyle="->", linewidth=1.2),
    )
    ax.text(0.69, 0.55, "DATA", fontsize=7, ha="center")
    ax.text(0.69, 0.43, "FEEDBACK", fontsize=7, ha="center")

def render_frame(frame_idx, save_path=None):
    s = stage_state(frame_idx)

    fig, ax = plt.subplots(figsize=(12.8, 7.2))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    # Title
    ax.text(
        0.04, 0.95,
        "열처리 공정 Digital Twin — 실제 공정과 가상 공정을 함께 이해하기",
        fontsize=18,
        fontweight="bold",
        va="top"
    )
    ax.text(
        0.04, 0.905,
        f"{s['name']}  |  Heat ID: HT-DEMO-001",
        fontsize=13,
        va="top"
    )

    # Process progress bar
    ax.add_patch(plt.Rectangle((0.04, 0.84), 0.92, 0.025, fill=False, linewidth=1.0))
    ax.add_patch(plt.Rectangle((0.04, 0.84), 0.92 * s["progress"], 0.025))
    ax.text(0.04, 0.875, "공정 진행률", fontsize=9)
    ax.text(0.96, 0.875, f"{s['progress']*100:4.0f}%", fontsize=9, ha="right")

    # Main furnace + DT graphic
    draw_furnace(ax, s)

    # Current values
    draw_bar(ax, 0.06, 0.22, 0.19, s["temp"] / 900.0, "온도", f"{s['temp']:.0f} °C")
    draw_bar(ax, 0.29, 0.22, 0.19, s["heater"] / 100.0, "히터 출력", f"{s['heater']:.0f} %")
    draw_bar(ax, 0.52, 0.22, 0.19, s["cp"] / 1.0, "탄소 포텐셜", f"{s['cp']:.2f}")
    draw_bar(ax, 0.75, 0.22, 0.19, s["energy"] / 200.0, "누적 에너지", f"{s['energy']:.0f} kWh")

    # Risk / outcome summary
    ax.text(0.06, 0.145, f"탄소배출 추정: {s['carbon']:.0f} kgCO₂ / Heat", fontsize=10)
    ax.text(0.36, 0.145, f"품질 위험: {s['quality_risk']*100:.0f}%", fontsize=10)
    ax.text(0.56, 0.145, f"이상 위험: {s['anomaly_risk']*100:.0f}%", fontsize=10)
    ax.text(0.75, 0.145, f"목표 온도: {s['target']:.0f} °C", fontsize=10)

    # Explainer message
    ax.add_patch(plt.Rectangle((0.04, 0.045), 0.92, 0.065, fill=False, linewidth=1.0))
    ax.text(0.06, 0.078, s["message"], fontsize=11, va="center")

    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        return None
    return fig, ax

# ------------------------------------------------------------
# 4) Generate 10 key-scene PNGs
# ------------------------------------------------------------
key_indices = np.linspace(0, N_FRAMES - 1, 10).astype(int)

for i, frame_idx in enumerate(key_indices, start=1):
    out = FRAME_DIR / f"scene_{i:02d}.png"
    render_frame(frame_idx, save_path=out)

# ------------------------------------------------------------
# 5) Create MP4
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12.8, 7.2))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

def update(frame_idx):
    ax.clear()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    s = stage_state(frame_idx)

    ax.text(
        0.04, 0.95,
        "열처리 공정 Digital Twin — 실제 공정과 가상 공정을 함께 이해하기",
        fontsize=18,
        fontweight="bold",
        va="top"
    )
    ax.text(
        0.04, 0.905,
        f"{s['name']}  |  Heat ID: HT-DEMO-001",
        fontsize=13,
        va="top"
    )

    ax.add_patch(plt.Rectangle((0.04, 0.84), 0.92, 0.025, fill=False, linewidth=1.0))
    ax.add_patch(plt.Rectangle((0.04, 0.84), 0.92 * s["progress"], 0.025))
    ax.text(0.04, 0.875, "공정 진행률", fontsize=9)
    ax.text(0.96, 0.875, f"{s['progress']*100:4.0f}%", fontsize=9, ha="right")

    draw_furnace(ax, s)

    draw_bar(ax, 0.06, 0.22, 0.19, s["temp"] / 900.0, "온도", f"{s['temp']:.0f} °C")
    draw_bar(ax, 0.29, 0.22, 0.19, s["heater"] / 100.0, "히터 출력", f"{s['heater']:.0f} %")
    draw_bar(ax, 0.52, 0.22, 0.19, s["cp"] / 1.0, "탄소 포텐셜", f"{s['cp']:.2f}")
    draw_bar(ax, 0.75, 0.22, 0.19, s["energy"] / 200.0, "누적 에너지", f"{s['energy']:.0f} kWh")

    ax.text(0.06, 0.145, f"탄소배출 추정: {s['carbon']:.0f} kgCO₂ / Heat", fontsize=10)
    ax.text(0.36, 0.145, f"품질 위험: {s['quality_risk']*100:.0f}%", fontsize=10)
    ax.text(0.56, 0.145, f"이상 위험: {s['anomaly_risk']*100:.0f}%", fontsize=10)
    ax.text(0.75, 0.145, f"목표 온도: {s['target']:.0f} °C", fontsize=10)

    ax.add_patch(plt.Rectangle((0.04, 0.045), 0.92, 0.065, fill=False, linewidth=1.0))
    ax.text(0.06, 0.078, s["message"], fontsize=11, va="center")
    return []

animation = FuncAnimation(fig, update, frames=N_FRAMES, interval=1000/FPS, blit=False)

mp4_path = ROOT / "heat_treatment_digital_twin_explainer.mp4"

writer = FFMpegWriter(
    fps=FPS,
    metadata={"title": "Heat Treatment Digital Twin Explainer"},
    bitrate=2200,
)

animation.save(str(mp4_path), writer=writer)
plt.close(fig)

print("생성 완료")
print("MP4:", mp4_path)
print("Key Scene PNG Folder:", FRAME_DIR)
'''

PROGRAM_PATH.write_text(PROGRAM, encoding="utf-8")
print("Saved:", PROGRAM_PATH)


In [ ]:
# 영상 생성 실행
%run /content/heat_treatment_digital_twin_layperson.py


In [ ]:
# MP4 재생
from IPython.display import Video, display

mp4_path = "/content/heat_treatment_digital_twin_demo/heat_treatment_digital_twin_explainer.mp4"
display(Video(mp4_path, embed=True, width=960))


In [ ]:
# 핵심 10개 장면 확인
from pathlib import Path
from IPython.display import Image, display

scene_dir = Path("/content/heat_treatment_digital_twin_demo/key_scenes")
for f in sorted(scene_dir.glob("*.png")):
    print(f.name)
    display(Image(filename=str(f), width=850))


In [ ]:
# MP4 파일 다운로드
from google.colab import files

files.download("/content/heat_treatment_digital_twin_demo/heat_treatment_digital_twin_explainer.mp4")


## 10개 장면의 의미

1. **공정 시작** — 실제 열처리 공정과 Digital Twin 연결
2. **제품 장입** — Heat ID와 제품·장입정보 연결
3. **승온 시작** — 히터 출력과 온도 상승
4. **고온 승온** — 에너지 소비와 미래 상태 예측
5. **유지** — 목표 온도에서 균일 열처리
6. **분위기 제어** — Carbon Potential과 가스 조건 관리
7. **확산** — 시간과 온도에 따른 품질 형성
8. **사전 냉각** — 냉각 조건과 품질 영향
9. **소입/급냉** — 변형·품질 위험 집중 관리
10. **공정 완료** — 실제 결과와 예측 결과를 비교해 다음 Heat 개선

### 이 영상은 무엇을 설명하는가?

이 영상의 Digital Twin은 단순 3D 모델이 아니라,

**실제 공정 상태 → 가상 모델 → 예측 → 비교 → 최적 조건 추천 → 실제 결과 피드백**

이라는 개념을 일반인이 이해할 수 있도록 단순화한 설명용 영상입니다.
